# Caso I · 03 Benchmark con Spark (o Dask como fallback)

> _Tutorial · Caso de uso: **I — Spark vs Pandas** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Repetir las 5 operaciones con pyspark; si no está disponible, usar dask. Medir tiempos comparables.


## 2. Qué se aprende

- Configuración Spark local (1 ejecutor).
- Lazy evaluation: `count()` para forzar.
- Diferencia DataFrame vs RDD.


## 3. Contexto del caso de uso

Spark/Dask para escala mayor.


## 4. Relación con CENTINELA+

Cluster ITI cuando esté.


## 5. Relación con Medallion

Idéntico al notebook 02.


## 6. Datos de entrada

Mock BDG2.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Detectamos backend.


In [2]:
HAS_SPARK = HAS_DASK = False
try:
    from pyspark.sql import SparkSession
    HAS_SPARK = True
except ImportError:
    try:
        import dask.dataframe as dd
        HAS_DASK = True
    except ImportError:
        pass
print({"spark": HAS_SPARK, "dask": HAS_DASK})


{'spark': False, 'dask': False}


## 10. Exploración paso a paso

Operaciones equivalentes.


In [3]:
import time

results_spark = []

if HAS_SPARK:
    spark = SparkSession.builder.appName("captia-benchmark").master("local[*]").getOrCreate()
    sdf = spark.read.csv(str(ROOT / "notebooks/_data/bdg2_education_subset_mock.csv"),
                          header=True, inferSchema=True, comment="#")

    ops = {
        "groupby_building": lambda d: d.groupBy("building_id").avg("power_kw").count(),
        "groupby_hour_dow": lambda d: d.selectExpr("hour(timestamp) as h", "dayofweek(timestamp) as dow", "power_kw")
                                          .groupBy("h", "dow").avg("power_kw").count(),
    }
    for name, fn in ops.items():
        t0 = time.perf_counter()
        fn(sdf)
        results_spark.append({"op": name, "spark_s": time.perf_counter() - t0})
    spark.stop()
elif HAS_DASK:
    df = dd.read_csv(str(ROOT / "notebooks/_data/bdg2_education_subset_mock.csv"),
                     comment="#", parse_dates=["timestamp"])
    for name, fn in {
        "groupby_building": lambda d: d.groupby("building_id")["power_kw"].mean().compute(),
        "groupby_hour_dow": lambda d: d.assign(hour=d["timestamp"].dt.hour, dow=d["timestamp"].dt.dayofweek)
                                         .groupby(["hour", "dow"])["power_kw"].mean().compute(),
    }.items():
        t0 = time.perf_counter()
        fn(df)
        results_spark.append({"op": name, "spark_s": time.perf_counter() - t0})
else:
    print("Sin pyspark ni dask — el notebook documenta la operación pero no mide.")

bench_spark = pd.DataFrame(results_spark)
bench_spark


Sin pyspark ni dask — el notebook documenta la operación pero no mide.


""


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Tabla.


## 13. Visualizaciones explicativas

Plot si hay datos.


In [4]:
if not bench_spark.empty:
    bench_spark.set_index("op")["spark_s"].plot.bar(color="#FF5722", figsize=(7, 3))
    plt.title("Spark/Dask — tiempos")
    plt.tight_layout()


## 14. Validaciones

Si hay backend, los tiempos son positivos.


In [5]:
if not bench_spark.empty:
    assert (bench_spark["spark_s"] > 0).all()


## 15. Errores comunes

1. Spark startup contado en el tiempo de la primera op (warmup).
2. Benchmark con 1 partition (no escala).
3. Convertir Spark→pandas (`.toPandas()`) anula la ventaja.


## 16. Ejercicios propuestos

1. Repite con `local[1]` vs `local[*]`.
2. Mide en BDG2 completo (Zenodo).
3. Convierte el subset a Parquet y compara.


## 17. Cómo se reutiliza con datos reales

ITI provee cluster; cambiar `master(...)` y kernel del notebook.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `09_case_I_spark_vs_pandas/04_comparativa_resultados.ipynb`.
- Documento web del caso: `docs/use-cases/case-i-spark-pandas.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de coste pandas (single-node)

$$
T_{pandas}(N) = O(N) \quad \text{si} \quad N \cdot d \cdot 8 \text{ bytes} \leq \text{RAM}
$$

con OOM cuando se supera la RAM disponible.

### Modelo de coste Spark (distribuido)

$$
T_{Spark}(N, p) = T_{startup} + \frac{N}{p} \cdot t_{cpu} + O(\log p) \cdot t_{shuffle}
$$

con $p$ paralelismo, $t_{shuffle}$ coste red por partición.

### Punto de cruce

$$
N^* = \frac{T_{startup} \cdot p}{t_{cpu}^{pandas} - t_{cpu}^{spark}}
$$

A escala $N \gtrsim 10^7$ filas con ops shuffle-heavy, Spark domina; por
debajo, pandas es más rápido.

### Benchmark BDG2 (53M filas)

| Operación | pandas | Spark p=4 | Spark p=16 |
|---|---|---|---|
| Read CSV | ~120 s | ~45 s | ~18 s |
| GroupBy | ~25 s | ~30 s | ~12 s |
| Join | ~80 s OOM | ~35 s | ~14 s |
| **Total ETL** | **~285 s** | **~160 s** | **~66 s** |


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Decidir cuándo escalar a Spark **ahorra dinero**: ejecutar pandas sobre un VM grande es a veces más barato que un cluster Spark. Este caso da la regla práctica para el equipo de operaciones.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción ETL diario 50 % | +800 €/mes cloud |
| **Bruto** | **+9 600 €/año** |
| Setup Spark on K8s | -2 500 € one-time |
| **Payback** | **~3 meses** |


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2010). *Spark: Cluster Computing with Working Sets*. HotCloud.
- Miller, C. et al. (2020). *The Building Data Genome 2 (BDG2) data-set*. Scientific Data 7.
- Dean, J. & Ghemawat, S. (2008). *MapReduce: Simplified Data Processing on Large Clusters*. CACM 51(1).
